In [6]:
import sys
sys.path.insert(0, "/home/xilinx/jupyter_notebooks/qick/qick_lib")
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import signal as scipy_signal
from qick import *

# -----------------------------------------------------------------------------
# 1. HARDWARE & INITIALIZATION SETUP
# -----------------------------------------------------------------------------
soc = QickSoc()
soccfg = soc

GEN_CH = 1
RO_CH = 0

gencfg = soccfg["gens"][GEN_CH]
samps_per_clk = gencfg["samps_per_clk"]
ENV_SR = gencfg["f_fabric"] * samps_per_clk * 1e6
ENV_MAXLEN = gencfg["maxlen"]

print(f"Generator {GEN_CH}: f_fabric={gencfg['f_fabric']:.3f} MHz, "
      f"samps_per_clk={samps_per_clk}, envelope sample rate={ENV_SR/1e9:.4f} GSPS")
print(f"Envelope memory available: {ENV_MAXLEN} samples")

# -----------------------------------------------------------------------------
# 2. SINGLE-TONE SERRODYNE STEP GENERATOR (continuous phase across steps)
#    FIX #3: n_samples is chosen from a small grid-aligned neighborhood
#    around the target duration, picking whichever candidate leaves the
#    smallest residual phase at the periodic-buffer wraparound point (i.e.
#    closest to landing on a whole number of cycles). This suppresses the
#    sidebands caused by mode="periodic" re-triggering on a non-continuous
#    phase boundary.
# -----------------------------------------------------------------------------
def best_n_samples(freq_hz, target_dur_s, sample_rate, samps_per_clk_, search_radius=3):
    target_n = int(round(target_dur_s * sample_rate))
    target_n = max(target_n, 1)
    # snap up to the fabric-cycle grid first
    pad0 = (-target_n) % samps_per_clk_
    target_n += pad0

    candidates = [target_n + k * samps_per_clk_
                  for k in range(-search_radius, search_radius + 1)]
    candidates = [n for n in candidates if n >= samps_per_clk_]

    if freq_hz == 0 or len(candidates) == 0:
        return target_n

    def residual(n):
        cycles = freq_hz * n / sample_rate
        frac = cycles - np.floor(cycles)
        return min(frac, 1.0 - frac)  # distance to nearest integer number of cycles

    return min(candidates, key=residual)


def serrodyne_tone(freq_hz, duration_sec, sample_rate, amplitude, phase0=0.0, width=1.0,
                    n_samples_override=None):
    if n_samples_override is not None:
        n_samples = max(1, int(n_samples_override))
    else:
        n_samples = max(1, int(round(float(duration_sec) * sample_rate)))
    dt = 1.0 / sample_rate
    t = np.arange(n_samples) * dt
    phase = 2 * np.pi * freq_hz * t + phase0
    y = float(amplitude) * scipy_signal.sawtooth(phase, width=width)
    phase_end = (phase0 + 2 * np.pi * freq_hz * n_samples * dt) % (2 * np.pi)
    return y, phase_end, n_samples

# -----------------------------------------------------------------------------
# 3. CHIRP DEFINITION -- each step's buffer is a 4-tone composite
#    (ratio-weighted durations). All 4 tones shift together as the chirp
#    offset sweeps from CHIRP_OFFSET_START_HZ to CHIRP_OFFSET_STOP_HZ.
# -----------------------------------------------------------------------------
CHIRP_SPAN_HZ = 350e6
CHIRP_OFFSET_STOP_HZ = 0.0                                       # ends AT the base tones
CHIRP_OFFSET_START_HZ = CHIRP_OFFSET_STOP_HZ - CHIRP_SPAN_HZ      # starts 350 MHz below them

AMPLITUDE = 0.5 #0.11

BASE_TONES_HZ = np.array([76.25e6, 0, -122.92e6+76.25e6, -147.82e6+76.25e6])
TONE_RATIOS = np.array([0.337, 0.167, 0.288, 0.208])

# BASE_TONES_HZ = np.array([0])
# TONE_RATIOS = np.array([1.0])

BASE_TONES_HZ *= -1
NUM_TONES = len(BASE_TONES_HZ)

TOTAL_SWEEP_S = 6e-3 # 6e-3

max_feasible_total_s = 0.95 * ENV_MAXLEN / ENV_SR
tone_fractions = TONE_RATIOS / TONE_RATIOS.sum()

# -----------------------------------------------------------------------------
# FIX #2: solve for the largest NUM_STEPS the envelope memory / fabric-cycle
# floor actually allows, instead of hand-picking a value and hoping it fits.
# Smaller NUM_STEPS => bigger CYCLE_S per step => larger chirp staircase
# error (scales as 1/NUM_STEPS^2) AND coarser sideband spacing. So we always
# want the ceiling this hardware budget permits.
# -----------------------------------------------------------------------------
def solve_max_num_steps(env_maxlen, env_sr, samps_per_clk_, tone_fractions_, safety=0.95):
    floor_samples = 3 * samps_per_clk_
    best_N = None
    N = 1
    while N <= 4000:
        max_feasible_total_s_ = safety * env_maxlen / env_sr
        cycle_s = max_feasible_total_s_ / (N + 1)
        tone_durations_s_ = tone_fractions_ * cycle_s

        piece_lens = []
        for dur in tone_durations_s_:
            n = int(round(dur * env_sr))
            n = max(n, 1)
            pad = (-n) % samps_per_clk_
            piece_lens.append(n + pad)
        samples_per_step_ = sum(piece_lens)
        total_samples_ = samples_per_step_ * (N + 1)  # sweep buffers + trap buffer (~same size)

        floor_ok = min(piece_lens) >= floor_samples
        mem_ok = total_samples_ <= env_maxlen

        if floor_ok and mem_ok:
            best_N = N
            N += 1
        else:
            break
    if best_N is None:
        raise ValueError("No feasible NUM_STEPS found -- TONE_RATIOS too extreme for this memory budget.")
    return best_N

NUM_STEPS = 40 #solve_max_num_steps(ENV_MAXLEN, ENV_SR, samps_per_clk, tone_fractions)
print(f"Solved max feasible NUM_STEPS = {NUM_STEPS} (was hand-set to 44 previously)")

STEP_HOLD_S = TOTAL_SWEEP_S / NUM_STEPS
STEP_HOLD_US = STEP_HOLD_S * 1e6

CYCLE_S = max_feasible_total_s / (NUM_STEPS + 1)   # one composite buffer's total duration
tone_durations_s = tone_fractions * CYCLE_S

print(f"Composite buffer cycle: {CYCLE_S*1e9:.2f} ns, held via mode='periodic' "
      f"for {STEP_HOLD_US:.2f} us per chirp step ({TOTAL_SWEEP_S*1e3:.3f} ms total sweep)")
for i, (bt, dur, r) in enumerate(zip(BASE_TONES_HZ, tone_durations_s, TONE_RATIOS)):
    print(f"  tone {i} ({bt/1e6:+.1f} MHz offset): ratio {r} -> requested {dur*1e9:.2f} ns")

def build_multitone_buffer(chirp_offset_hz, phase0):
    """One composite envelope: all NUM_TONES tones, each shifted by the same
    chirp_offset_hz and ratio-weighted in duration, concatenated.
    Sample count per tone is chosen (within a small grid-aligned window) to
    minimize the phase residual at the mode="periodic" wraparound point."""
    i_pieces, q_pieces = [], []
    phase = phase0
    for base_tone, dur in zip(BASE_TONES_HZ, tone_durations_s):
        f = chirp_offset_hz + base_tone
        n_best = best_n_samples(f, dur, ENV_SR, samps_per_clk)
        y, phase, n_samples = serrodyne_tone(f, dur, ENV_SR, amplitude=AMPLITUDE, phase0=phase,
                                              n_samples_override=n_best)
        pad = (-len(y)) % samps_per_clk
        if pad:
            y = np.concatenate([y, np.zeros(pad)])
        i_pieces.append(np.round(y * maxv).astype(np.int16))
        q_pieces.append(np.zeros(len(y), dtype=np.int16))
    return np.concatenate(i_pieces), np.concatenate(q_pieces), phase, [len(p) for p in i_pieces]

# -----------------------------------------------------------------------------
# FIX #1: midpoint chirp sampling instead of edge sampling.
# Each held step now represents the chirp's average frequency over that
# dwell interval (evaluated at the interval midpoint), rather than the
# frequency at the interval's start. This makes the instantaneous phase
# error within a step symmetric (+-beta*tau^2/8) instead of one-sided
# (0 to beta*tau^2/2) -- a 4x reduction in peak phase error for free.
# The very last step is snapped exactly to CHIRP_OFFSET_STOP_HZ so the
# hand-off into the trap buffer stays phase-continuous at the target freq.
# -----------------------------------------------------------------------------
step_width_hz = (CHIRP_OFFSET_STOP_HZ - CHIRP_OFFSET_START_HZ) / NUM_STEPS
chirp_offsets_hz = CHIRP_OFFSET_START_HZ + (np.arange(NUM_STEPS) + 0.5) * step_width_hz
chirp_offsets_hz[-1] = CHIRP_OFFSET_STOP_HZ  # land exactly on target freq before trapping

maxv = soccfg.get_maxv(GEN_CH)

idata_list = []
qdata_list = []
phase = 0.0
for offset in chirp_offsets_hz:
    idata, qdata, phase, piece_lens = build_multitone_buffer(offset, phase)
    idata_list.append(idata)
    qdata_list.append(qdata)

samples_per_step = len(idata_list[0])
lengths = [len(x) for x in idata_list]
print(set(lengths))
total_samples = samples_per_step * NUM_STEPS
print(f"Per-step composite buffer length: {samples_per_step} samples "
      f"(tone slices: {piece_lens}, smallest = {min(piece_lens)/samps_per_clk:.1f} fabric cycles)")
print(f"Total envelope samples (sweep): {total_samples} / {ENV_MAXLEN} available")
assert total_samples <= ENV_MAXLEN
assert samples_per_step % samps_per_clk == 0
assert min(piece_lens) >= 3 * samps_per_clk, (
    f"Smallest tone slice is only {min(piece_lens)} samples "
    f"({min(piece_lens)/samps_per_clk:.1f} fabric cycles) -- hardware needs >= 3. "
    f"Reduce NUM_STEPS, or make TONE_RATIOS less extreme."
)

# -----------------------------------------------------------------------------
# 3b. TRAP BUFFER: same ratio-weighted 4-tone composite, at the FINAL chirp
#     frequency, phase-continuous with the last sweep step.
# -----------------------------------------------------------------------------
trap_idata, trap_qdata, phase, trap_piece_lens = build_multitone_buffer(CHIRP_OFFSET_STOP_HZ, phase)
print(f"Trap buffer: {len(trap_idata)} samples (tone slices: {trap_piece_lens})")

total_with_trap = total_samples + len(trap_idata)
print(f"Total envelope samples (sweep + trap): {total_with_trap} / {ENV_MAXLEN} available")
assert total_with_trap <= ENV_MAXLEN, (
    f"Sweep + trap buffers exceed memory: need {total_with_trap}, "
    f"have {ENV_MAXLEN}. Reduce NUM_STEPS or CYCLE_S."
)

# -----------------------------------------------------------------------------
# 4. PROGRAM: sweep as before, then ONE final pulse on the concatenated trap
#    buffer with mode="periodic" -- hardware loops it forever on its own.
# -----------------------------------------------------------------------------
class RepeatedStepSerrodyneProgram(RAveragerProgram):
    def initialize(self):
        cfg = self.cfg
        res_ch = cfg["res_ch"]

        self.declare_gen(ch=res_ch, nqz=1)

        for i, (idata_step, qdata_step) in enumerate(zip(cfg["idata_list"], cfg["qdata_list"])):
            self.add_envelope(ch=res_ch, name=f"serr_{i}", idata=idata_step, qdata=qdata_step)

        self.add_envelope(ch=res_ch, name="trap_wfm", idata=cfg["trap_idata"], qdata=cfg["trap_qdata"])

        self.set_pulse_registers(
            ch=res_ch,
            style="arb",
            freq=0,
            phase=0,
            gain=cfg["gain"],
            waveform="serr_0",
            outsel="input",
            mode="periodic",
        )

        self.r_rp = self.ch_page(res_ch)
        self.r_addr = self.sreg(res_ch, "addr")
        self.addr_step = cfg["samples_per_step"] // self.soccfg["gens"][res_ch]["samps_per_clk"]

        self.synci(200)

    def body(self):
        res_ch = self.cfg["res_ch"]
        step_cycles = self.us2cycles(self.cfg["step_hold_us"])

        self.trigger(pins=[0])
        self.pulse(ch=res_ch, t='auto')
        self.sync_all(step_cycles)

    def update(self):
        self.mathi(self.r_rp, self.r_addr, self.r_addr, '+', self.addr_step)

    def make_program(self):
        """Standard RAveragerProgram sweep (expts x reps), with a single
        trapping pulse appended after the loop -- no loop construct needed
        for trapping, since mode="periodic" makes the hardware repeat it
        on its own once triggered."""
        p = self
        rcount = 13
        rii = 14
        rjj = 15

        p.initialize()
        p.regwi(0, rcount, 0)
        p.regwi(0, rii, self.cfg['expts'] - 1)
        p.label("LOOP_I")
        p.regwi(0, rjj, self.cfg['reps'] - 1)
        p.label("LOOP_J")
        p.body()
        p.mathi(0, rcount, rcount, "+", 1)
        p.memwi(0, rcount, self.COUNTER_ADDR)
        p.loopnz(0, rjj, 'LOOP_J')
        p.update()
        p.loopnz(0, rii, "LOOP_I")

        # --- trapping: one pulse, periodic buffer, then end(). The DAC keeps
        # cycling the 4 concatenated tones forever regardless of what the
        # tProc does after this -- including after it hits end() and halts.
        res_ch = self.cfg["res_ch"]
        p.set_pulse_registers(
            ch=res_ch, style="arb", freq=0, phase=0, gain=self.cfg["gain"],
            waveform="trap_wfm", outsel="input", mode="periodic",
        )
        p.trigger(pins=[0])
        p.pulse(ch=res_ch, t='auto')
        p.end()

# -----------------------------------------------------------------------------
# 5. EXECUTION
# -----------------------------------------------------------------------------
config = {
    "res_ch": GEN_CH,
    "reps": 1,
    "expts": NUM_STEPS,
    "idata_list": idata_list,
    "qdata_list": qdata_list,
    "trap_idata": trap_idata,
    "trap_qdata": trap_qdata,
    "samples_per_step": samples_per_step,
    "step_hold_us": STEP_HOLD_US,
    "gain": 32767,
}

prog = RepeatedStepSerrodyneProgram(soccfg, config)
# prog.run(soc) #, start_src="external"
prog.run(soc, start_src="external")
print(f"Running on hardware — {NUM_STEPS} sweep steps x {STEP_HOLD_US:.2f} us "
      f"= {NUM_STEPS*STEP_HOLD_US*1e-3:.3f} ms sweep, then trapping "
      f"(4 tones, periodic, indefinitely until soc.reset_gens()).")


Generator 1: f_fabric=614.400 MHz, samps_per_clk=16, envelope sample rate=9.8304 GSPS
Envelope memory available: 65536 samples
Solved max feasible NUM_STEPS = 40 (was hand-set to 44 previously)
Composite buffer cycle: 154.47 ns, held via mode='periodic' for 150.00 us per chirp step (6.000 ms total sweep)
  tone 0 (-76.2 MHz offset): ratio 0.337 -> requested 52.06 ns
  tone 1 (-0.0 MHz offset): ratio 0.167 -> requested 25.80 ns
  tone 2 (+46.7 MHz offset): ratio 0.288 -> requested 44.49 ns
  tone 3 (+71.6 MHz offset): ratio 0.208 -> requested 32.13 ns
{1536, 1568, 1600, 1664, 1472, 1440, 1504, 1632, 1552, 1520, 1584, 1456, 1392}
Per-step composite buffer length: 1552 samples (tone slices: [512, 256, 416, 272], smallest = 16.0 fabric cycles)
Total envelope samples (sweep): 62080 / 65536 available
Trap buffer: 1456 samples (tone slices: [512, 256, 416, 272])
Total envelope samples (sweep + trap): 63536 / 65536 available
Running on hardware — 40 sweep steps x 150.00 us = 6.000 ms sweep, th

In [7]:
soc.reset_gens()

Generator 1: f_fabric=614.400 MHz, samps_per_clk=16, envelope sample rate=9.8304 GSPS
Envelope memory: 65536 samples

Max feasible NUM_STEPS  = 77
Samples per step        = 832 (fixed, all steps identical)
Tone slice lengths      = [272, 144, 240, 176] samples
Composite buffer cycle  = 81.20 ns
Step hold time          = 77.922 us
Total sweep             = 6.000 ms
Total samples (sweep)   = 64064 / 65536
Total samples (+trap)   = 64896 / 65536

Checking phase residuals with FIXED buffer lengths...
Max residual across all steps/tones: 0.4996 cycles
  NOTE: residuals > 0.05 are unavoidable with these frequencies
  and this buffer size. The phase error per replay is bounded by
  2*pi*0.500 = 3.139 rad
  This is acceptable for a chirp — the phase ramps continuously.

Building 77 envelopes (832 samples each, fixed)...
All 77 envelopes: 832 samples ✓
Trap buffer: 832 samples ✓
Total (sweep+trap): 64896 / 65536 ✓

Running: 77 steps x 77.922 us = 6.000 ms sweep
Then: trap buffer loops at +0.0 M